In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

from sklearn.model_selection import train_test_split

# import function to perform linear regression
from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor

# import StandardScaler to perform scaling
from sklearn.preprocessing import StandardScaler 

# import function to perform GridSearchCV
from sklearn.model_selection import GridSearchCV

from sklearn.model_selection import TimeSeriesSplit

import warnings
warnings.filterwarnings('ignore')

# Loading the dataset

In [ ]:
df = pd.read_excel("Online Retail.xlsx")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

# Checking for missing values

In [ ]:
df.isnull().sum()

# Checking for Duplicate values

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

# Handling missing values

In [ ]:
df = df.dropna(subset = ["Description"])

In [ ]:
df = df.drop(columns = "CustomerID")

# Creating sales column

In [ ]:
df["Sales"] = (df["Quantity"]*df["UnitPrice"])

In [ ]:
df.head()

# Removing Cancelled orders

In [ ]:
df = df[df["InvoiceNo"].astype(str).str.startswith("C")==False]

# Removing Invalid Transactions

In [ ]:
df = df[(df["Quantity"]>0) & (df["UnitPrice"]>0)]

# Aggregrating Daily Sales

In [ ]:
daily_sales = (df.groupby(df['InvoiceDate'].dt.date)['Sales'].sum().reset_index())
daily_sales.columns = ["Date","Sales"]
daily_sales["Date"] = pd.to_datetime(daily_sales["Date"])

# Sales Trend Analysis

### Daily Sales Trend

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(daily_sales['Date'],daily_sales['Sales'])

plt.title("Daily Sales Trend")

plt.xlabel("Date")

plt.ylabel("Sales")

plt.show()

### Monthly Sales Trend

In [ ]:
monthly_sales = daily_sales.groupby(daily_sales['Date'].dt.to_period('M'))['Sales'].sum()

monthly_sales.plot(figsize=(15,6))
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.show()

### Distribution of Sales

In [ ]:
sns.histplot(daily_sales["Sales"],kde = True)
plt.title("Distribution of Sales")
plt.show()

# Time Series Based feature Engineering

In [ ]:
daily_sales["Month"] = daily_sales["Date"].dt.month

daily_sales["Day"] = daily_sales["Date"].dt.day

daily_sales["DayOfWeek"] = daily_sales["Date"].dt.dayofweek

daily_sales["Quarters"] = daily_sales["Date"].dt.quarter

# Week Day Analysis

In [ ]:
daily_sales['DayName'] = daily_sales['DayOfWeek'].map({0:'Monday',
                                                       1:'Tuesday',
                                                       2:'Wednesday',
                                                       3:'Thursday',
                                                       4:'Friday',
                                                       5:'Saturday',
                                                       6:'Sunday'})

weekday_sales = daily_sales.groupby('DayName')['Sales'].mean()

plt.figure(figsize=(15,6))

weekday_sales.plot(kind='bar')

plt.xticks(rotation=45)

plt.title('Average Sales by Day')

plt.xlabel("Days of Week")

plt.ylabel("Sales")

plt.show()

# Lag features

In [ ]:
daily_sales["Lag1"] = daily_sales["Sales"].shift(1) # Previous day's Sale

daily_sales["Lag7"] = daily_sales["Sales"].shift(7) # Sales from 7 days ago

daily_sales['Lag14'] = daily_sales['Sales'].shift(14) # sales from 14 days ago

daily_sales["Lag30"] = daily_sales["Sales"].shift(30) # Sales from 30 days ago

# Rolling Mean (Moving Average)

In [ ]:
daily_sales['Rolling_Mean_7'] = daily_sales['Sales'].rolling(7).mean() # Average Sales of last 7 days

daily_sales['Rolling_Mean_14'] = daily_sales['Sales'].rolling(14).mean() # Average Sales of last 14 days

daily_sales["Rolling_Mean_30"] = daily_sales["Sales"].rolling(30).mean() # Average Sales of last 30 days

In [ ]:
daily_sales.dropna(inplace = True)

# Correlation Heatmap

In [ ]:
plt.figure(figsize = (15,6))

sns.heatmap(daily_sales.corr(numeric_only = True), annot = True)

plt.title("Correlation Heatmap")

plt.show()

# Selecting Feature and target Variable

In [ ]:
X = daily_sales.drop(["Date","Sales","DayName"], axis = 1)
y = daily_sales["Sales"]

# Train Test Split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size = 0.2, shuffle = False) # Past → predicts → Future (preserve order of time)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# Time Series Cross Validation

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

# Linear Regression

In [ ]:
linear_model = LinearRegression()

linear_model.fit(X_train_scaled, y_train)

linear_pred = linear_model.predict(X_test_scaled)

In [ ]:
linear_train_pred = linear_model.predict(X_train_scaled)

print("Train R2:", r2_score(y_train, linear_train_pred))

print("Test R2:",r2_score(y_test, linear_pred))

# Random Forest

In [ ]:
tuned_params=[{'n_estimators':[100,200,300],'max_depth':[3,5,7,10],'min_samples_split' : [5, 10, 20],'min_samples_leaf': [2,5,10]}]
rf =RandomForestRegressor(random_state=100)
rf_grid=GridSearchCV(estimator=rf,param_grid=tuned_params,cv=tscv,n_jobs=-1,scoring = "neg_mean_squared_error").fit(X_train,y_train)
print('Best Parameters:',rf_grid.best_params_,'\n')
rf_best_model = rf_grid.best_estimator_

In [ ]:
rf_pred =rf_best_model.predict(X_test)

In [ ]:
train_pred = rf_best_model.predict(X_train)

print("Train R2:", r2_score(y_train, train_pred))

print("Test R2:",r2_score(y_test, rf_pred))

# Important Features

In [ ]:
important_features=pd.DataFrame({'Features':X_train.columns,'Importance':rf_best_model.feature_importances_})
importance_df=important_features.sort_values('Importance',ascending=False)
importance_df

In [ ]:
# To draw horizontal barchart

plt.figure(figsize=(15,6))

plt.barh(importance_df['Features'],importance_df['Importance'])

plt.title('Random Forest Feature Importance')

plt.xlabel('Importance')

plt.show()

# Evaluation Metrices

In [ ]:
def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    return mae,rmse,r2

# Evaluate Models

In [ ]:
linear_mae, linear_rmse, linear_r2 = evaluate(y_test, linear_pred)

rf_mae, rf_rmse, rf_r2 = evaluate(y_test, rf_pred)

In [ ]:
results = pd.DataFrame({'Model':['Linear Regression','Random Forest'],'MAE':[linear_mae,rf_mae],'RMSE':[linear_rmse,rf_rmse],'R2':[linear_r2,rf_r2]})

results

In [ ]:
plt.figure(figsize=(15,6))

test_dates = daily_sales.loc[y_test.index, 'Date']

plt.plot(test_dates,y_test,label='Actual Sales')

plt.plot(test_dates,linear_pred,label='Predicted Sales')

plt.legend()

plt.title("Actual vs Predicted Sales")

plt.xticks(rotation=45)

plt.show()

# Future Forecasting 

In [ ]:
future_features = X_test.tail(30) # Take last 30 rows from the test dataset

future_features_scaled = scaler.transform(future_features)

future_forecast = linear_model.predict(future_features_scaled)

In [ ]:
# to find the last date from the data and move one day ahead and create 30 future dates

future_dates = pd.date_range(start = daily_sales["Date"].max() + pd.Timedelta(days = 1), periods = 30)

forecast_df = pd.DataFrame({"Date" : future_dates, "Forecasted_Sales" : future_forecast})

forecast_df

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(forecast_df['Date'],forecast_df['Forecasted_Sales'],marker='o')

plt.title('Next 30 Days Sales Forecast')

plt.xlabel('Date')

plt.ylabel('Forecasted Sales')

plt.xticks(rotation=45)

plt.show()

# Buisness Summary

In [ ]:
business_summary = pd.DataFrame({'Metric':['Average Daily Sales',
                                           'Maximum Daily Sales',
                                           'Minimum Daily Sales',
                                           'Forecast Average Sales'],
                                 'Value':[daily_sales['Sales'].mean(),
                                          daily_sales['Sales'].max(),
                                          daily_sales['Sales'].min(),
                                          forecast_df['Forecasted_Sales'].mean()]})

business_summary